Creating pyspark sesssion!

In [1]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName('Week5_DataCleaning') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'Spark Version: {spark.version}')
print('SparkSession ready ✅')

Spark Version: 4.0.3
SparkSession ready ✅


Generating a dataset

In [3]:
import random
import string
from datetime import date, timedelta

random.seed(42)

regions       = ['West', 'East', 'North', 'South']
categories    = ['Electronics', 'Clothing', 'Grocery', 'Furniture', 'Sports']
cities        = ['Mumbai', 'Delhi', 'Jaipur', 'Bangalore', 'Chennai',
                 'Hyderabad', 'Pune', 'Kolkata', 'Ahmedabad', 'Surat']
subscriptions = ['Premium', 'Basic', 'Free']
statuses      = ['Active', 'Inactive', None, 'Pending', None]
store_ids     = [f'STR_{i:03d}' for i in range(1, 21)]

def rand_date():
    start = date(2023, 1, 1)
    return str(start + timedelta(days=random.randint(0, 729)))

def rand_email(uid):
    return None if random.random() < 0.08 else f'user{uid}@mail.com'

def rand_username():
    return '' if random.random() < 0.06 else 'user_' + ''.join(random.choices(string.ascii_lowercase, k=5))

def rand_price():
    return None if random.random() < 0.07 else round(random.uniform(50, 5000), 2)

rows = []
for i in range(1, 601):
    uid   = random.randint(1, 520) if random.random() > 0.08 else random.randint(1, 50)
    tdate = rand_date()
    rows.append((
        uid,
        tdate,
        random.choice(regions),
        random.choice(categories),
        round(random.uniform(100, 10000), 2),
        random.choice(cities),
        random.randint(16, 65),
        random.choice(subscriptions),
        rand_email(uid),
        rand_username(),
        rand_price(),
        f'{tdate}T{random.randint(0,23):02d}:{random.randint(0,59):02d}:00',
        random.choice(store_ids),
        random.choice(statuses)
    ))

schema = StructType([
    StructField('user_id',          IntegerType(), True),
    StructField('transaction_date', StringType(),  True),
    StructField('region',           StringType(),  True),
    StructField('product_category', StringType(),  True),
    StructField('sale_amount',      DoubleType(),  True),
    StructField('city',             StringType(),  True),
    StructField('age',              IntegerType(), True),
    StructField('subscription',     StringType(),  True),
    StructField('email',            StringType(),  True),
    StructField('username',         StringType(),  True),
    StructField('price',            DoubleType(),  True),
    StructField('raw_timestamp',    StringType(),  True),
    StructField('store_id',         StringType(),  True),
    StructField('status',           StringType(),  True)
])

df = spark.createDataFrame(rows, schema=schema)
print(f'Rows: {df.count()} | Columns: {len(df.columns)}')
df.show(5, truncate=False)

Rows: 600 | Columns: 14
+-------+----------------+------+----------------+-----------+---------+---+------------+----------------+----------+-------+-------------------+--------+-------+
|user_id|transaction_date|region|product_category|sale_amount|city     |age|subscription|email           |username  |price  |raw_timestamp      |store_id|status |
+-------+----------------+------+----------------+-----------+---------+---+------------+----------------+----------+-------+-------------------+--------+-------+
|26     |2023-10-09      |East  |Clothing        |1481.43    |Delhi    |59 |Free        |user26@mail.com |user_kafna|3266.93|2023-10-09T17:26:00|STR_008 |Pending|
|7      |2023-06-13      |South |Grocery         |2850.93    |Bangalore|64 |Basic       |user7@mail.com  |user_jigbl|4615.36|2023-06-13T02:35:00|STR_010 |NULL   |
|371    |2024-08-14      |East  |Electronics     |553.66     |Bangalore|65 |Basic       |user371@mail.com|user_wjlve|3367.37|2024-08-14T22:59:00|STR_003 |NULL   

### Question 3 -  Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [4]:
print(f'Before removing duplicate: {df.count()} rows')

df_nodupe = df.dropDuplicates(['user_id', 'transaction_date'])

print(f'After removing duplicates:  {df_nodupe.count()} rows')
print(f'Duplicates removed: {df.count() - df_nodupe.count()}')

Before removing duplicate: 600 rows
After removing duplicates:  600 rows
Duplicates removed: 0


### Q4 -Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [5]:
result_q4 = (df.filter(F.col('region') == 'West').groupBy('product_category')
.agg(F.round(F.avg('sale_amount'), 2).alias('avg_sale_amount'))
.orderBy('avg_sale_amount', ascending=False))

result_q4.show()

+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
|          Sports|        5397.24|
|         Grocery|        4835.43|
|       Furniture|        4736.58|
|        Clothing|        4715.85|
|     Electronics|        4116.73|
+----------------+---------------+



In [6]:
# Validation — manually verify one category
df.filter((F.col('region') == 'West') & (F.col('product_category') == 'Sports')).agg(F.round(F.avg('sale_amount'), 2).alias('manual_avg'),
F.count('*').alias('row_count')).show()

+----------+---------+
|manual_avg|row_count|
+----------+---------+
|   5397.24|       32|
+----------+---------+



## Insight -
- Sports has the highest average sale amount (₹5397.24) in the West region,
  suggesting premium buying behaviour in that category.

- Electronics has the lowest avg (₹4116.73) despite typically being a
  high-value category — could indicate more low-ticket electronic items
  being sold in the West.

- All 5 categories are represented in the West region with avg sales
  between ₹4100–₹5400 — a relatively uniform spread.

- Filtering before groupBy reduced data volume before aggregation —
  this minimises shuffle overhead.

### Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

In [7]:
# Check nulls before
print('Null count in status before fill:')
df.select(F.count(F.when(F.col('status').isNull(), 1)).alias('null_status')).show()

# Fill nulls with 'Unknown'
df_filled = df.na.fill({'status': 'Unknown'})

# Check nulls after
print('Null count in status after fill:')
df_filled.select(F.count(F.when(F.col('status').isNull(), 1)).alias('null_status')).show()

# Confirm distinct values
print('Distinct status values after fill:')
df_filled.select('status').distinct().show()

Null count in status before fill:
+-----------+
|null_status|
+-----------+
|        249|
+-----------+

Null count in status after fill:
+-----------+
|null_status|
+-----------+
|          0|
+-----------+

Distinct status values after fill:
+--------+
|  status|
+--------+
| Unknown|
|  Active|
|Inactive|
| Pending|
+--------+



## Insights
- 249 out of 600 rows (approx. 41.5%) had null status values —
  a significant portion that would silently skew any analysis.

- After .na.fill(), null count dropped to 0 — all rows
  now have a valid status value.

- 4 distinct values

### Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [10]:
result_q6 = (df.groupBy('city').agg(F.count('*').alias('record_count'))
.filter(F.col('record_count') > 60).orderBy('record_count', ascending=False))

result_q6.show()

+------+------------+
|  city|record_count|
+------+------------+
| Delhi|          84|
|  Pune|          67|
|Jaipur|          64|
+------+------------+



 # NOTE:
I changed the original threshold of 100 to 60 to suit our
synthetic dataset of 600 rows across 10 cities (~60 rows/city avg).


## Insights

- Only 3 out of 10 cities crossed the threshold of 60 records:
  Delhi (84), Pune (67), and Jaipur (64).

- Delhi has the highest transaction volume — most statistically
  reliable city for further analysis.

- 7 cities fell below the threshold — not enough data points
  to draw m

### Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [11]:
result_q8 = df.filter((F.col('age').between(18, 30)) &
 (F.col('subscription') == 'Premium'))

print(f'Matching rows: {result_q8.count()}')
result_q8.select('user_id', 'age', 'subscription', 'city').show(10)

Matching rows: 49
+-------+---+------------+---------+
|user_id|age|subscription|     city|
+-------+---+------------+---------+
|    411| 25|     Premium|   Mumbai|
|     48| 25|     Premium|     Pune|
|     75| 19|     Premium|Ahmedabad|
|    240| 20|     Premium|  Kolkata|
|    144| 24|     Premium|Hyderabad|
|    240| 22|     Premium|    Surat|
|    484| 22|     Premium|   Jaipur|
|     14| 25|     Premium|  Kolkata|
|    490| 26|     Premium|   Mumbai|
|     11| 28|     Premium|    Surat|
+-------+---+------------+---------+
only showing top 10 rows


## Insights

- 49 out of 600 rows (approx. 8.2%) match both conditions —
  age 18-30 AND Premium subscription.

- .between(18, 30) is inclusive on both ends — equivalent
  to age >= 18 AND age <= 30.

- This segment (young Premium users) is typically the most
  valuable for businesses — high engagement, long customer
  lifetime value potential.

### Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [13]:
df_q10 = df.withColumn('event_time',F.col('raw_timestamp')
.cast(TimestampType())).drop('raw_timestamp')

df_q10.select('user_id', 'transaction_date', 'event_time').show(5, truncate=False)
df_q10.printSchema()

+-------+----------------+-------------------+
|user_id|transaction_date|event_time         |
+-------+----------------+-------------------+
|26     |2023-10-09      |2023-10-09 17:26:00|
|7      |2023-06-13      |2023-06-13 02:35:00|
|371    |2024-08-14      |2024-08-14 22:59:00|
|251    |2023-06-17      |2023-06-17 22:20:00|
|470    |2023-05-27      |2023-05-27 04:40:00|
+-------+----------------+-------------------+
only showing top 5 rows
root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timest

## Insights

- raw_timestamp (StringType) was successfully cast to
  event_time (TimestampType) — confirmed in printSchema().

- withColumn() both creates and replaces columns in one call.
  The original raw_timestamp was dropped to avoid redundancy.

- Casting to TimestampType unlocks time-based operations like
  hour(), dayofweek(), and window functions — not possible
  on a plain string column.

- event_time is now the last column

### Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [14]:
#Identify bad rows first

print('Null emails:', df.filter(F.col('email').isNull()).count())
print('Empty usernames:', df.filter(F.col('username') == '').count())
print('Total bad rows:', df.filter(F.col('email').isNull() |
(F.col('username') == '')).count())

print(f'\nBefore cleaning: {df.count()} rows')

# Remove bad rows
df_q12 = df.filter(F.col('email').isNotNull() &
 (F.col('username') != '')
)

print(f'After cleaning: {df_q12.count()} rows')

# Validation — confirm no nulls or empty strings remain
print('\nValidation:')
print('Null emails remaining:', df_q12.filter(F.col('email').isNull()).count())
print('Empty usernames remaining:', df_q12.filter(F.col('username') == '').count())

Null emails: 35
Empty usernames: 27
Total bad rows: 60

Before cleaning: 600 rows
After cleaning: 540 rows

Validation:
Null emails remaining: 0
Empty usernames remaining: 0


## Insights

- 35 rows had null emails, 27 had empty usernames —
  60 total bad rows (10% of dataset) were removed.

- After cleaning: 540 rows remain — all with valid
  email AND non-empty username, confirmed by validation.

- Null and empty string are different things — a column
  can be non-null but still empty (''). Always check both
  when validating string identity fields.

- isNull() handles None/null values, != '' handles
  empty strings — two separate conditions combined with &.

### Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [16]:
result_q13 = df.agg(
F.round(F.min('price'),  2).alias('min_price'),
F.round(F.max('price'),  2).alias('max_price'),
F.round(F.avg('price'),  2).alias('mean_price'),
F.count('price').alias('non_null_count'),
F.count('*').alias('total_count'))

result_q13.show()

# Validation — verify min and max manually
print('Validation:')

df.select('price').filter(F.col('price').isNotNull()).orderBy('price').show(3)
df.select(F.col('price')).orderBy('price').show(3)
df.select(F.col('price')).orderBy(F.col('price').desc()).show(3)

+---------+---------+----------+--------------+-----------+
|min_price|max_price|mean_price|non_null_count|total_count|
+---------+---------+----------+--------------+-----------+
|     62.7|  4999.54|   2585.21|           562|        600|
+---------+---------+----------+--------------+-----------+

Validation:
+------+
| price|
+------+
|  62.7|
| 91.17|
|109.31|
+------+
only showing top 3 rows
+-----+
|price|
+-----+
| NULL|
| NULL|
| NULL|
+-----+
only showing top 3 rows
+-------+
|  price|
+-------+
|4999.54|
|4980.17|
|4962.81|
+-------+
only showing top 3 rows


## Insights

- .agg() without groupBy computes stats across the entire DataFrame.

- 562 out of 600 prices are non-null (38 nulls) — confirmed by
  comparing non_null_count vs total_count. These nulls were
  silently excluded from min/max/mean calculations.

- Price range: ₹62.70 (min) to ₹4999.54 (max), mean ₹2585.21 —
  fairly centered, suggesting uniform distribution in our dataset.

- Always include both count('col') and count('*') in aggregations —
  the difference reveals how many nulls exist in that column.

### Q15: Write a final processing pipeline that:

Filters out duplicates.

Fills null prices with 0.

Groups by store_id to calculate total revenue

In [17]:
pipeline_result = (df.dropDuplicates(['user_id', 'transaction_date'])   # Remove duplicates
.na.fill({'price': 0.0})                                                # Fill null prices with 0
.groupBy('store_id')                                                    # Group by store_id, calculate total revenue
.agg(F.round(F.sum('price'), 2).alias('total_revenue'),
F.count('*').alias('transaction_count'))
.orderBy('total_revenue', ascending=False))

pipeline_result.show(20)

# Validation — confirm no nulls in price before pipeline
print('Null prices after fill:',
    df.na.fill({'price': 0.0}).filter(F.col('price').isNull()).count()
)

+--------+-------------+-----------------+
|store_id|total_revenue|transaction_count|
+--------+-------------+-----------------+
| STR_002|    107462.23|               39|
| STR_004|     90389.27|               32|
| STR_016|     86340.35|               37|
| STR_012|     86242.16|               29|
| STR_006|     81169.28|               34|
| STR_014|     77731.95|               31|
| STR_010|     76231.49|               34|
| STR_011|     75794.22|               32|
| STR_009|     75663.88|               35|
| STR_005|     74760.09|               28|
| STR_020|     73810.92|               35|
| STR_018|     73522.44|               28|
| STR_003|     72030.63|               28|
| STR_017|     71307.75|               33|
| STR_013|     68136.54|               29|
| STR_015|     62731.81|               23|
| STR_019|     58855.54|               30|
| STR_001|     55997.92|               21|
| STR_007|     48938.53|               23|
| STR_008|     35768.92|               19|
+--------+-

## Insights

- Pipeline successfully chained 3 cleaning + aggregation steps
  in one expression — this is the Spark ETL pattern.

- STR_002 is the top performing store (₹107,462.23 revenue,
  39 transactions) — highest both in revenue and volume.

- STR_008 is the lowest performer (₹35,768.92, 19 transactions) —
  nearly 3x less revenue than STR_002.

- Filling null prices with 0 before sum() ensures no revenue
  is silently excluded from totals — validation confirmed
  0 nulls remaining after fill.

- transaction_count alongside total_revenue gives a fuller
  picture — a store with high revenue but low count means
  high-value transactions, not just high volume.